# **Coleta de Sumulas da CBF**

## **1. Configuracao do Ambiente**

In [ ]:
import re
import time
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import requests
import urllib3
from urllib3.exceptions import InsecureRequestWarning

urllib3.disable_warnings(InsecureRequestWarning)

## **2. Configuracao**

In [ ]:
BASE_URL = "https://www.cbf.com.br"
API_URL = f"{BASE_URL}/api/cbf/jogos/campeonato"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_PATH = PROJECT_ROOT / "data"
RAW_DATA_PATH = DATA_PATH / "raw"
MATCH_REPORTS_PATH = RAW_DATA_PATH / "match-reports"
VIZ_PATH = PROJECT_ROOT / "viz"

YEARS = list(range(2020, 2027))
COMPETITION_IDS = {
    2020: "12464",
    2021: "12487",
    2022: "12518",
    2023: "12555",
    2024: "12584",
    2025: "12606",
    2026: "1260611",
}
MAX_ROUNDS = 38
REQUEST_DELAY = 0.2

for path in [MATCH_REPORTS_PATH, VIZ_PATH]:
    path.mkdir(parents=True, exist_ok=True)

## **3. Funcoes Utilitarias**

In [ ]:
def normalize_text(value: str) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(char for char in text if not unicodedata.combining(char))
    return text.casefold().strip()


def slugify(value: str) -> str:
    text = normalize_text(value)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    return text.strip("-") or "sem-nome"


def safe_int(value, default: int = 0) -> int:
    try:
        return int(str(value).strip())
    except (TypeError, ValueError):
        return default


def fetch_json(session: requests.Session, url: str, max_retries: int = 3) -> Optional[dict]:
    last_error = None
    for attempt in range(max_retries):
        try:
            response = session.get(url, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as error:
            last_error = error
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
    raise RuntimeError(f"Falha em {url}") from last_error

## **4. Coleta de Documentos**

In [ ]:
def flatten_documents(payload: dict, year: int, competition_id: str, round_number: int) -> List[Dict[str, str]]:
    rows = []
    for group in payload.get("jogos", []) or []:
        for game in group.get("jogo", []) or []:
            base = {
                "year": year,
                "competition_id": competition_id,
                "round": safe_int(game.get("rodada"), round_number),
                "game_id": game.get("id_jogo"),
                "game_number": safe_int(game.get("num_jogo")),
                "home_team": game.get("mandante", {}).get("nome"),
                "away_team": game.get("visitante", {}).get("nome"),
                "match_date": str(game.get("data", "")).strip(),
                "match_time": str(game.get("hora", "")).strip(),
                "venue": game.get("local"),
            }
            for document in game.get("documentos", []) or []:
                title = document.get("title", "")
                rows.append({
                    **base,
                    "document_title": title,
                    "document_title_normalized": normalize_text(title),
                    "document_url": document.get("url", ""),
                })
    return rows


def collect_documents() -> pd.DataFrame:
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.verify = False
    rows = []
    for year in YEARS:
        competition_id = COMPETITION_IDS[year]
        for round_number in range(1, MAX_ROUNDS + 1):
            url = f"{API_URL}/{competition_id}/rodada/{round_number}/fase"
            payload = fetch_json(session, url)
            rows.extend(flatten_documents(payload, year, competition_id, round_number))
            time.sleep(REQUEST_DELAY)
    data = pd.DataFrame(rows)
    if not data.empty:
        data = data.sort_values(["year", "round", "game_number", "document_title"]).reset_index(drop=True)
    return data

In [ ]:
documents = collect_documents()
documents_index_path = MATCH_REPORTS_PATH / "cbf_documents_2020_2026.csv"
documents.to_csv(documents_index_path, index=False)
documents.head()

## **5. Download das Sumulas**

In [ ]:
def build_match_report_path(row: pd.Series) -> Path:
    home = slugify(row["home_team"])
    away = slugify(row["away_team"])
    year = int(row["year"])
    round_number = int(row["round"])
    game_number = int(row["game_number"])
    game_id = row["game_id"]
    filename = f"{year}_r{round_number:02d}_j{game_number:03d}_{home}_x_{away}_{game_id}_sumula.pdf"
    return MATCH_REPORTS_PATH / str(year) / f"round-{round_number:02d}" / filename


def download_pdf(session: requests.Session, url: str, path: Path, max_retries: int = 3) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return "exists"
    temp_path = path.with_suffix(path.suffix + ".tmp")
    last_error = None
    for attempt in range(max_retries):
        try:
            with session.get(url, timeout=60, stream=True) as response:
                response.raise_for_status()
                with temp_path.open("wb") as file:
                    for chunk in response.iter_content(chunk_size=8192):
                        if chunk:
                            file.write(chunk)
            temp_path.replace(path)
            return "downloaded"
        except requests.RequestException as error:
            last_error = error
            if temp_path.exists():
                temp_path.unlink()
            if attempt < max_retries - 1:
                time.sleep((attempt + 1) * 2)
    raise RuntimeError(f"Falha no download {url}") from last_error


def collect_match_reports(documents_df: pd.DataFrame) -> pd.DataFrame:
    reports = documents_df[documents_df["document_title_normalized"].eq("sumula")].copy()
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0"})
    session.verify = False
    reports["file_path"] = ""
    reports["download_status"] = ""
    for index, row in reports.iterrows():
        file_path = build_match_report_path(row)
        status = download_pdf(session, row["document_url"], file_path)
        reports.loc[index, "file_path"] = str(file_path)
        reports.loc[index, "download_status"] = status
        time.sleep(REQUEST_DELAY)
    return reports.reset_index(drop=True)

In [ ]:
match_reports = collect_match_reports(documents)
match_reports_index_path = MATCH_REPORTS_PATH / "match_reports_index.csv"
match_reports.to_csv(match_reports_index_path, index=False)
match_reports["download_status"].value_counts(dropna=False)